# PediaVision: Pediatric Skin Analysis
## CNN + Q&A + Treatment Recommendations

basically the pipeline is:
1. upload image -> CNN gives initial guess
2. ask user some follow up questions (age, symptoms, etc)
3. refined diagnosis with confidence
4. treatment recs pulled from sephora/amazon data

datasets:
- HAM10000 (skin lesion images, ~10k)
- ACNE04 (acne severity)
- sephora dataset for product recs
- used llama for the Q&A part

## setup

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q pillow matplotlib seaborn scikit-learn
!pip install -q kaggle pandas numpy opencv-python
!pip install -q efficientnet_pytorch
!pip install -q gradio

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import json
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using: {device}")
print(f"gpu: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (running on cpu)'}")

## download datasets

In [ ]:
from google.colab import files
print("upload kaggle.json:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ham10000
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d data/ham10000/

# acne04
!kaggle datasets download -d imtkaggleteam/acne-computer-vision
!unzip -q acne-computer-vision.zip -d data/acne04/

# sephora
!kaggle datasets download -d raghadalharbi/all-products-available-on-sephora-website
!unzip -q all-products-available-on-sephora-website.zip -d data/sephora/

print("done")

## data loading + preprocessing

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
            label = self.labels[idx]
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:
            print(f"couldn't load {self.image_paths[idx]}: {e}")
            return torch.zeros(3, 224, 224), label


def load_ham10000():
    df = pd.read_csv('data/ham10000/HAM10000_metadata.csv')
    
    # readable names
    dx_to_name = {
        'akiec': 'Actinic_Keratoses',
        'bcc': 'Basal_Cell_Carcinoma',
        'bkl': 'Benign_Keratosis',
        'df': 'Dermatofibroma',
        'mel': 'Melanoma',
        'nv': 'Melanocytic_Nevi',
        'vasc': 'Vascular_Lesions'
    }
    df['diagnosis'] = df['dx'].map(dx_to_name)
    
    # find each image (split across two folders)
    paths = []
    for img_id in df['image_id']:
        p1 = f'data/ham10000/HAM10000_images_part_1/{img_id}.jpg'
        p2 = f'data/ham10000/HAM10000_images_part_2/{img_id}.jpg'
        if os.path.exists(p1):
            paths.append(p1)
        elif os.path.exists(p2):
            paths.append(p2)
        else:
            paths.append(None)
    
    df['image_path'] = paths
    df = df[df['image_path'].notna()]
    return df[['image_path', 'diagnosis', 'age', 'sex', 'localization']]


def load_acne04():
    rows = []
    for base in ['data/acne04/', 'data/acne04/images/']:
        for severity in ['normal', 'mild', 'moderate', 'severe']:
            folder = os.path.join(base, severity)
            if not os.path.exists(folder):
                continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.png', '.jpeg')):
                    rows.append({
                        'image_path': os.path.join(folder, fname),
                        'diagnosis': f'Acne_{severity.capitalize()}',
                        'age': None,
                        'sex': None,
                        'localization': 'face'
                    })
    return pd.DataFrame(rows)


print("loading ham10000...")
ham_df = load_ham10000()
print(f"ham10000: {len(ham_df)} images")

print("loading acne04...")
acne_df = load_acne04()
print(f"acne04: {len(acne_df)} images")

combined_df = pd.concat([ham_df, acne_df], ignore_index=True)
print(f"\ntotal: {len(combined_df)} images")
print(combined_df['diagnosis'].value_counts())

In [ ]:
# augmentations for training
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# no augmentation for val
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

le = LabelEncoder()
combined_df['label_encoded'] = le.fit_transform(combined_df['diagnosis'])
np.save('label_classes.npy', le.classes_)

print(f"classes ({len(le.classes_)}): {list(le.classes_)}")

train_df, val_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df['label_encoded'],
    random_state=42
)

train_dataset = SkinDataset(train_df['image_path'].values, train_df['label_encoded'].values, transform=train_transform)
val_dataset   = SkinDataset(val_df['image_path'].values,   val_df['label_encoded'].values,   transform=val_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"train: {len(train_dataset)} | val: {len(val_dataset)}")

## model

using efficientnet-b3 pretrained, froze most of the early layers and replaced the classifier head

In [ ]:
class SkinCNN(nn.Module):
    def __init__(self, num_classes):
        super(SkinCNN, self).__init__()
        self.backbone = models.efficientnet_b3(pretrained=True)
        
        # freeze early layers, only fine-tune the last ~30
        for param in list(self.backbone.parameters())[:-30]:
            param.requires_grad = False
        
        in_features = self.backbone.classifier[1].in_features
        
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(p=0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)


num_classes = len(le.classes_)
model = SkinCNN(num_classes).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"total params:     {total_params:,}")
print(f"trainable params: {trainable_params:,}")
print(f"frozen params:    {total_params - trainable_params:,}")
print(f"num classes:      {num_classes}")

## training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, preds = out.max(1)
        total   += labels.size(0)
        correct += preds.eq(labels).sum().item()
        
        if i % 50 == 0:
            print(f'  batch [{i}/{len(loader)}] loss: {loss.item():.4f}')
    
    return total_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out   = model(imgs)
            loss  = criterion(out, labels)
            probs = torch.softmax(out, dim=1)
            
            total_loss += loss.item()
            _, preds = out.max(1)
            total   += labels.size(0)
            correct += preds.eq(labels).sum().item()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels, all_probs


NUM_EPOCHS   = 20
best_val_acc = 0
history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    print(f"\nepoch [{epoch+1}/{NUM_EPOCHS}]")
    print("-" * 50)
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_preds, val_labels, val_probs = validate(model, val_loader, criterion)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"train loss: {train_loss:.4f} | train acc: {train_acc:.2f}%")
    print(f"val loss:   {val_loss:.4f} | val acc:   {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'label_encoder': le
        }, 'best_skin_model.pth')
        print(f"  -> new best saved ({val_acc:.2f}%)")
    
    scheduler.step()
    
    if epoch > 10 and val_acc < best_val_acc - 5:
        print("early stopping")
        break

print(f"\ndone. best val acc: {best_val_acc:.2f}%")

## evaluation

In [ ]:
checkpoint = torch.load('best_skin_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

val_loss, val_acc, preds, labels, probs = validate(model, val_loader, criterion)

print(f"val accuracy: {val_acc:.2f}%")
print(f"val loss:     {val_loss:.4f}")

print("\n" + classification_report(labels, preds, target_names=le.classes_, digits=3))

# confusion matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('confusion matrix')
plt.ylabel('true')
plt.xlabel('predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'],   label='val')
ax1.set_title('loss')
ax1.set_xlabel('epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='train')
ax2.plot(history['val_acc'],   label='val')
ax2.set_title('accuracy (%)')
ax2.set_xlabel('epoch')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Q&A system (llm for clarification)

after the cnn gives an initial prediction, we ask the user some follow-up questions to refine it. using tinyllama since it actually fits in colab

In [ ]:
print("loading llm...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# tinyllama fits in colab, bigger models oom
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("loaded")


class QASystem:
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model     = model
        
        self.questions = {
            'age':                "What is the patient's age?",
            'duration':           "How long has this been present? (days/weeks/months)",
            'symptoms':           "Any symptoms? (itching, pain, burning, etc.)",
            'previous_treatment': "Tried any treatments already?",
            'family_history':     "Family history of skin conditions?",
            'location':           "Where on the body?",
            'changes':            "Has it changed recently?",
            'allergies':          "Any known allergies?"
        }
    
    def get_questions(self, diagnosis, confidence):
        # always ask these
        to_ask = ['age', 'duration', 'symptoms', 'location']
        
        # condition-specific extras
        if 'Acne' in diagnosis:
            to_ask += ['previous_treatment']
        elif 'Melanoma' in diagnosis or 'Carcinoma' in diagnosis:
            to_ask += ['family_history', 'changes']
        elif 'Keratosis' in diagnosis:
            to_ask += ['changes', 'previous_treatment']
        
        # if confidence is low, ask more
        if confidence < 0.7:
            to_ask += ['allergies', 'family_history']
        
        to_ask = list(set(to_ask))
        return [self.questions[q] for q in to_ask if q in self.questions]
    
    def analyze(self, diagnosis, qa_pairs):
        prompt = f"""You are a pediatric dermatology assistant.

Initial diagnosis: {diagnosis}

Patient info:
"""
        for q, a in qa_pairs.items():
            prompt += f"- {q}: {a}\n"
        
        prompt += """\nGiven this, provide:
1. Updated confidence (0-100%)
2. Any additional observations
3. Risk factors
4. Urgency (low/medium/high)

JSON format:"""
        
        inputs  = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        return {'analysis': response, 'qa_summary': qa_pairs}


qa_system = QASystem(tokenizer, llm)
print("qa system ready")

## treatment recommendations

hardcoded treatment plans per condition + pulls product names from sephora dataset

In [ ]:
try:
    sephora_df = pd.read_csv('data/sephora/sephora_website_dataset.csv')
    print(f"loaded {len(sephora_df)} sephora products")
except:
    print("sephora data not found, skipping product recs")
    sephora_df = pd.DataFrame()


class TreatmentSystem:
    def __init__(self, product_df=None):
        self.products = product_df
        
        self.plans = {
            'Acne_Normal': {
                'urgency': 'low',
                'see_doctor': 'just routine checkups',
                'otc': ['gentle cleanser (cetaphil, cerave)', 'moisturizer with SPF 30+', 'vitamin C serum'],
                'routine': ['AM: cleanse -> vit C -> moisturizer -> sunscreen', 'PM: cleanse -> moisturize'],
                'notes': 'pretty much just maintenance skincare',
                'red_flags': []
            },
            'Acne_Mild': {
                'urgency': 'low',
                'see_doctor': 'if no improvement after 6-8 weeks',
                'otc': ['benzoyl peroxide 2.5% cleanser', 'salicylic acid spot treatment', 'non-comedogenic moisturizer', 'oil-free SPF 30+'],
                'routine': ['AM: gentle cleanse -> moisturize -> sunscreen', 'PM: BP cleanser -> spot treatment -> moisturize'],
                'notes': 'takes 4-6 weeks to see results, dont pick at it, change pillowcase weekly',
                'red_flags': ['painful cysts', 'spreading', 'scarring']
            },
            'Acne_Moderate': {
                'urgency': 'medium',
                'see_doctor': 'worth seeing a derm',
                'otc': ['salicylic acid cleanser 2%', 'benzoyl peroxide 5-10%', 'differin (adapalene 0.1%)', 'niacinamide serum', 'oil-free moisturizer'],
                'routine': ['AM: SA cleanser -> niacinamide -> BP -> moisturize -> SPF 50', 'PM: gentle cleanse -> adapalene -> moisturize'],
                'notes': 'takes 8-12 weeks. use retinoid at night only. sunscreen non-negotiable',
                'red_flags': ['deep cysts', 'no improvement after 12 weeks', 'bad scarring']
            },
            'Acne_Severe': {
                'urgency': 'high',
                'see_doctor': 'see a dermatologist asap',
                'otc': ['OTC isnt going to cut it', 'gentle cleanser only for now', 'fragrance-free moisturizer'],
                'routine': ['just gentle cleansing until derm appointment', 'do not pick'],
                'notes': 'needs prescription - probably antibiotics or accutane. dont mess around with this one',
                'red_flags': ['already severe, needs professional help']
            },
            'Melanoma': {
                'urgency': 'CRITICAL',
                'see_doctor': 'go to a dermatologist within 24-48 hours',
                'otc': ['no OTC treatment - needs medical eval immediately'],
                'routine': ['protect from sun', 'document changes with photos'],
                'notes': 'melanoma is serious but very treatable if caught early. do not wait',
                'red_flags': ['asymmetry', 'irregular border', 'multiple colors', 'diameter >6mm', 'changing over time']
            },
            'Melanocytic_Nevi': {
                'urgency': 'low',
                'see_doctor': 'annual skin check is good practice',
                'otc': ['SPF 50+ daily', 'just monitor it'],
                'routine': ['daily sunscreen', 'monthly self-check'],
                'notes': 'most moles are fine. watch ABCDE: asymmetry, border, color, diameter, evolution',
                'red_flags': ['rapid growth', 'color change', 'bleeding', 'itching']
            },
            'Actinic_Keratoses': {
                'urgency': 'medium',
                'see_doctor': 'see derm - these are pre-cancerous',
                'otc': ['strict SPF 50+', 'protective clothing'],
                'routine': ['minimize sun exposure', 'hats + long sleeves outside'],
                'notes': 'AKs can progress if left untreated. derm will probably do cryo or topical treatment',
                'red_flags': ['growing fast', 'bleeding', 'not healing']
            },
            'Basal_Cell_Carcinoma': {
                'urgency': 'high',
                'see_doctor': 'schedule derm within 1-2 weeks',
                'otc': ['no OTC - needs treatment'],
                'routine': ['sun protection', 'monitor'],
                'notes': 'most common skin cancer but rarely spreads. very treatable',
                'red_flags': ['non-healing sore', 'growing', 'bleeding']
            },
            'Benign_Keratosis': {
                'urgency': 'low',
                'see_doctor': 'optional - removal is cosmetic',
                'otc': ['moisturizer if dry', 'sunscreen'],
                'routine': ['normal skincare'],
                'notes': 'benign, no treatment needed unless it bothers you',
                'red_flags': ['sudden change', 'bleeding', 'pain']
            },
            'Dermatofibroma': {
                'urgency': 'low',
                'see_doctor': 'only if it bothers you',
                'otc': ['nothing needed'],
                'routine': ['normal skincare'],
                'notes': 'benign, feels firm, totally harmless',
                'red_flags': ['rapid growth', 'pain', 'color change']
            },
            'Vascular_Lesions': {
                'urgency': 'low',
                'see_doctor': 'if changing or cosmetically bothersome',
                'otc': ['gentle skincare', 'sunscreen'],
                'routine': ['avoid trauma to the area'],
                'notes': 'usually benign, laser treatment available if cosmetic removal wanted',
                'red_flags': ['growing fast', 'bleeding a lot']
            }
        }
    
    def get_plan(self, diagnosis, qa_context=None):
        plan = self.plans.get(diagnosis, self.plans['Acne_Normal']).copy()
        
        if qa_context:
            plan['personalization'] = self._personalize(plan, qa_context)
        
        if self.products is not None and len(self.products) > 0:
            plan['products'] = self._find_products(diagnosis)
        
        return plan
    
    def _personalize(self, plan, qa_context):
        notes = []
        if 'age' in qa_context:
            age_str = qa_context['age'].lower()
            if any(x in age_str for x in ['child', 'teen', '<18', 'under 18']):
                notes.append("pediatric patient - check dosing with peds derm")
        if 'allergies' in qa_context and qa_context['allergies'].lower() not in ['no', 'none', 'n/a']:
            notes.append("allergies reported - double check ingredient lists")
        if 'previous_treatment' in qa_context and 'yes' in qa_context['previous_treatment'].lower():
            notes.append("previous treatment noted - may need to pivot approach")
        return " | ".join(notes) if notes else "standard protocol"
    
    def _find_products(self, diagnosis):
        # simple keyword match
        keywords = {
            'Acne':      ['acne', 'blemish', 'oil control', 'clarifying'],
            'Melanoma':  ['sunscreen', 'spf', 'sun protection'],
            'Keratosis': ['exfoliant', 'aha', 'bha', 'retinol']
        }
        terms = []
        for key in keywords:
            if key in diagnosis:
                terms.extend(keywords[key])
        if not terms:
            return []
        
        found = []
        for _, row in self.products.head(100).iterrows():
            name = str(row.get('name', '')).lower()
            if any(t in name for t in terms):
                found.append(row.get('name', 'unknown'))
                if len(found) >= 3:
                    break
        return found


treatment_system = TreatmentSystem(sephora_df if len(sephora_df) > 0 else None)
print("treatment system ready")

## full pipeline

In [ ]:
class Pipeline:
    def __init__(self, cnn, qa, treatment, label_encoder):
        self.cnn       = cnn
        self.qa        = qa
        self.treatment = treatment
        self.le        = label_encoder
        self.transform = val_transform
    
    def predict_image(self, image_path):
        img    = Image.open(image_path).convert('RGB')
        tensor = self.transform(img).unsqueeze(0).to(device)
        
        self.cnn.eval()
        with torch.no_grad():
            out   = self.cnn(tensor)
            probs = torch.softmax(out, dim=1)
            conf, pred = probs.max(1)
        
        diagnosis = self.le.classes_[pred.item()]
        top3_probs, top3_idx = probs.topk(3, dim=1)
        top3 = [(self.le.classes_[i.item()], p.item()) for i, p in zip(top3_idx[0], top3_probs[0])]
        
        return {'diagnosis': diagnosis, 'confidence': conf.item(), 'top3': top3}
    
    def run(self, image_path, interactive=True):
        print("\n" + "="*60)
        print("PEDIAVISION ANALYSIS")
        print("="*60)
        
        # stage 1: cnn
        print("\nanalyzing image...")
        result = self.predict_image(image_path)
        print(f"diagnosis:  {result['diagnosis']}")
        print(f"confidence: {result['confidence']*100:.1f}%")
        print("\ntop 3:")
        for i, (d, p) in enumerate(result['top3'], 1):
            print(f"  {i}. {d}: {p*100:.1f}%")
        
        # stage 2: Q&A
        questions  = self.qa.get_questions(result['diagnosis'], result['confidence'])
        qa_answers = {}
        
        if interactive:
            print(f"\n{len(questions)} follow-up questions:\n")
            for i, q in enumerate(questions, 1):
                qa_answers[q] = input(f"{i}. {q}\n> ")
        else:
            print("\nquestions that would be asked:")
            for i, q in enumerate(questions, 1):
                print(f"  {i}. {q}")
            qa_answers = {q: "[demo]" for q in questions}
        
        # stage 3: refine
        refined = self.qa.analyze(result['diagnosis'], qa_answers)
        result['qa'] = refined
        result['qa_answers'] = qa_answers
        
        # stage 4: treatment
        plan = self.treatment.get_plan(result['diagnosis'], qa_answers)
        
        print("\n" + "="*60)
        print("RESULTS")
        print("="*60)
        print(f"\ndiagnosis: {result['diagnosis']}")
        print(f"urgency:   {plan['urgency'].upper()}")
        print(f"\nwhen to see doctor: {plan['see_doctor']}")
        
        print("\ntreatments:")
        for t in plan['otc']:
            print(f"  - {t}")
        
        print("\nroutine:")
        for step in plan['routine']:
            print(f"  - {step}")
        
        print(f"\nnotes: {plan['notes']}")
        
        if plan['red_flags']:
            print("\nwarning signs to watch for:")
            for flag in plan['red_flags']:
                print(f"  - {flag}")
        
        if 'personalization' in plan:
            print(f"\npersonalized notes: {plan['personalization']}")
        
        if 'products' in plan and plan['products']:
            print("\nproduct suggestions:")
            for p in plan['products']:
                print(f"  - {p}")
        
        return {'result': result, 'plan': plan}


pipeline = Pipeline(model, qa_system, treatment_system, le)
print("pipeline ready")

## test

In [ ]:
test_path  = val_df.iloc[0]['image_path']
true_label = val_df.iloc[0]['diagnosis']

print(f"image: {test_path}")
print(f"true label: {true_label}")

output = pipeline.run(test_path, interactive=False)

img = Image.open(test_path)
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title(f"predicted: {output['result']['diagnosis']}\nconfidence: {output['result']['confidence']*100:.1f}%")
plt.axis('off')
plt.tight_layout()
plt.show()

## save + export

In [ ]:
# save pytorch model
torch.save({
    'model_state_dict': model.state_dict(),
    'label_encoder': le,
    'classes': le.classes_,
    'num_classes': len(le.classes_)
}, 'pediavision_final.pth')

np.save('label_classes.npy', le.classes_)
pd.DataFrame(history).to_csv('training_history.csv', index=False)

with open('treatment_database.json', 'w') as f:
    json.dump(treatment_system.plans, f, indent=2)

print("saved: pediavision_final.pth, label_classes.npy, training_history.csv, treatment_database.json")

# convert to coreml for iOS
try:
    import coremltools as ct
    
    example_input = torch.rand(1, 3, 224, 224).to(device)
    traced = torch.jit.trace(model, example_input)
    
    mlmodel = ct.convert(
        traced,
        inputs=[ct.ImageType(name="input_image", shape=(1, 3, 224, 224))],
        classifier_config=ct.ClassifierConfig(list(le.classes_))
    )
    mlmodel.save("PediaVision.mlmodel")
    print("saved: PediaVision.mlmodel")
    
except ImportError:
    print("coremltools not found, run: pip install coremltools")

## coreml export + download

In [ ]:
!pip install -q coremltools
import coremltools as ct

# reload best checkpoint
checkpoint = torch.load('best_skin_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model = model.cpu()

print(f"loaded model (val acc: {checkpoint['val_acc']:.2f}%)")

# trace
example = torch.rand(1, 3, 224, 224)
with torch.no_grad():
    traced = torch.jit.trace(model, example)

# convert
mlmodel = ct.convert(
    traced,
    inputs=[ct.ImageType(
        name="input_image",
        shape=(1, 3, 224, 224),
        scale=1/255.0,
        bias=[0, 0, 0],
        color_layout='RGB'
    )],
    classifier_config=ct.ClassifierConfig(
        class_labels=list(le.classes_),
        predicted_feature_name="classLabel",
        predicted_probabilities_output="classProbabilities"
    )
)

# metadata
mlmodel.author            = "PediaVision"
mlmodel.short_description = "Pediatric skin condition classifier (HAM10000 + ACNE04)"
mlmodel.version           = "1.0"
mlmodel.input_description["input_image"]          = "224x224 RGB skin image"
mlmodel.output_description["classLabel"]          = "predicted condition"
mlmodel.output_description["classProbabilities"]  = "per-class confidence scores"

mlmodel.save("PediaVision.mlmodel")
print("saved PediaVision.mlmodel")

# model info json
model_info = {
    "model_name": "PediaVision",
    "version": "1.0",
    "classes": list(le.classes_),
    "num_classes": len(le.classes_),
    "input_size": [224, 224],
    "accuracy": float(checkpoint['val_acc']),
    "architecture": "EfficientNet-B3",
    "datasets": ["HAM10000", "ACNE04"]
}
with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

# check files
needed = ['PediaVision.mlmodel', 'model_info.json', 'treatment_database.json',
          'best_skin_model.pth', 'label_classes.npy', 'training_history.csv']

print("\nfiles:")
for f in needed:
    if os.path.exists(f):
        mb = os.path.getsize(f) / (1024*1024)
        print(f"  {f} ({mb:.2f} MB)")
    else:
        print(f"  MISSING: {f}")

# download
from google.colab import files
for f in needed:
    if os.path.exists(f):
        files.download(f)

## metrics for sccur presentation

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    labels, preds, average=None, labels=range(num_classes)
)

perf_df = pd.DataFrame({
    'class':     le.classes_,
    'precision': precision,
    'recall':    recall,
    'f1':        f1,
    'support':   support
}).sort_values('f1', ascending=False)

# bar chart
fig, ax = plt.subplots(figsize=(12, 7))
x     = np.arange(len(perf_df))
width = 0.25

ax.bar(x - width, perf_df['precision'], width, label='precision', alpha=0.8)
ax.bar(x,         perf_df['recall'],    width, label='recall',    alpha=0.8)
ax.bar(x + width, perf_df['f1'],        width, label='f1',        alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(perf_df['class'], rotation=45, ha='right')
ax.set_ylabel('score')
ax.set_title('per-class performance')
ax.legend()
ax.set_ylim([0, 1.1])
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('per_class_performance.png', dpi=300, bbox_inches='tight')
plt.show()

perf_df.to_csv('per_class_metrics.csv', index=False)

# summary
print(f"overall accuracy:  {best_val_acc:.2f}%")
print(f"num classes:       {num_classes}")
print(f"train samples:     {len(train_dataset):,}")
print(f"val samples:       {len(val_dataset):,}")
print(f"architecture:      EfficientNet-B3")
print(f"total params:      {total_params:,}")

metrics = {
    'accuracy': float(best_val_acc),
    'num_classes': int(num_classes),
    'train_samples': len(train_dataset),
    'val_samples': len(val_dataset),
    'total_params': int(total_params),
    'datasets': {'ham10000': len(ham_df), 'acne04': len(acne_df), 'sephora': len(sephora_df)}
}
with open('research_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nsaved per_class_performance.png, per_class_metrics.csv, research_metrics.json")

## xcode integration notes

files to drag into xcode:
- `PediaVision.mlmodel`
- `model_info.json`
- `treatment_database.json`

check "copy items if needed" and add to your app target

use the SkinAnalysisManager.swift to call the model, add camera permissions to Info.plist, then run on device

backup files (don't need in xcode):
- `best_skin_model.pth` - pytorch weights if you want to retrain
- `training_history.csv` - for the graphs in the presentation
- `research_metrics.json` - numbers for the slides